### Clean Events (dedupe + normalization)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
bronze_table = "subscription_platform.bronze.subscription_events"

silver_events_clean_table_path = "s3://subscription-revenue-platform/silver/subscription_events_clean/"
silver_events_clean_checkpoint_path = "s3://subscription-revenue-platform/silver/silver_events_clean_checkpoints/"
silver_events_clean_schema_path = f"{silver_events_clean_checkpoint_path}/schema"

In [0]:
catalog = "subscription_platform"
schema = "silver"
silver_events_clean_table = f"{catalog}.{schema}.subscription_events_clean"

In [0]:
bronze_df = (
    spark.readStream
    .option("schemaTrackingLocation", silver_events_clean_schema_path)
    .table(bronze_table)
)

In [0]:
events_clean = (
    bronze_df
    .withWatermark("event_time", "2 days")
    .dropDuplicates(["event_id"])
)

In [0]:
events_clean = (
    events_clean
    .withColumn("event_type", lower(col("event_type")))
    .withColumn("event_date", to_date("event_time"))
)

In [0]:
events_clean = events_clean.withColumn(
    "status",
    when(col("event_type") == "subscription_created", "active")
    .when(col("event_type") == "plan_changed", "active")
    .when(col("event_type") == "subscription_reactivated", "active")
    .when(col("event_type") == "subscription_cancelled", "cancelled")
)

In [0]:
(
    events_clean.writeStream
    .format("delta")
    .option("checkpointLocation", silver_events_clean_checkpoint_path)
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(silver_events_clean_table)
)

In [0]:
%sql

SELECT *
FROM subscription_platform.silver.subscription_events_clean
LIMIT 20;

event_id,event_type,customer_id,subscription_id,plan_id,amount,currency,event_time,ingested_at,source,processed_at,source_file,event_date,status
evt_44838c402c6b407da984a0345502d347,subscription_cancelled,cust_503,sub_1003,basic,1000.0,INR,2026-04-06T18:04:17.656Z,2026-04-09T18:04:17.656Z,billing_lambda,2026-04-10T21:03:44.988Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-06/evt_44838c402c6b407da984a0345502d347.json,2026-04-06,cancelled
evt_d621b41e7791437aa6ce46fb43ed846f,plan_changed,cust_503,sub_1003,pro,2000.0,INR,2026-04-09T18:04:24.986Z,2026-04-09T18:04:24.986Z,billing_lambda,2026-04-10T21:03:44.988Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_d621b41e7791437aa6ce46fb43ed846f.json,2026-04-09,active
evt_b5e71536d22e467b9dad09f94d7f85e6,subscription_cancelled,cust_501,sub_1001,basic,1000.0,INR,2026-04-09T18:04:24.986Z,2026-04-09T18:04:24.986Z,billing_lambda,2026-04-10T21:03:44.988Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_b5e71536d22e467b9dad09f94d7f85e6.json,2026-04-09,cancelled
evt_ce1596a3ed2b4221ab6c18c586ee9fd6,subscription_cancelled,cust_503,sub_1003,basic,1000.0,INR,2026-04-09T18:04:14.732Z,2026-04-09T18:04:14.732Z,billing_lambda,2026-04-10T21:03:44.988Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_ce1596a3ed2b4221ab6c18c586ee9fd6.json,2026-04-09,cancelled
evt_0cff3647b74e4b3196231d1c4e48fcc3,subscription_cancelled,cust_501,sub_1001,basic,1000.0,INR,2026-04-09T18:04:24.986Z,2026-04-09T18:04:24.986Z,billing_lambda,2026-04-10T21:03:44.988Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_0cff3647b74e4b3196231d1c4e48fcc3.json,2026-04-09,cancelled
evt_2765d7d00c254b068b8b3fcd071d5f1b,subscription_reactivated,cust_501,sub_1001,basic,1000.0,INR,2026-04-09T18:03:46.374Z,2026-04-09T18:03:46.374Z,billing_lambda,2026-04-10T21:03:44.988Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_2765d7d00c254b068b8b3fcd071d5f1b.json,2026-04-09,active
evt_dc44f42c9ac7449dbe0d422a2e7c51fa,subscription_reactivated,cust_501,sub_1001,basic,1000.0,INR,2026-04-09T18:04:20.449Z,2026-04-09T18:04:20.449Z,billing_lambda,2026-04-10T21:03:44.988Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_dc44f42c9ac7449dbe0d422a2e7c51fa.json,2026-04-09,active
evt_cc3668af98604eab8d47c59142e8fff8,subscription_reactivated,cust_503,sub_1003,basic,1000.0,INR,2026-04-09T18:04:24.986Z,2026-04-09T18:04:24.986Z,billing_lambda,2026-04-10T21:03:44.988Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_cc3668af98604eab8d47c59142e8fff8.json,2026-04-09,active
evt_404316929a924ea699996616c56fb71c,subscription_created,cust_501,sub_1001,basic,1000.0,INR,2026-04-09T18:04:14.732Z,2026-04-09T18:04:14.732Z,billing_lambda,2026-04-10T21:03:44.988Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_404316929a924ea699996616c56fb71c.json,2026-04-09,active
evt_7024658bf3a44af9adde25d76268206c,subscription_reactivated,cust_503,sub_1003,basic,1000.0,INR,2026-04-09T18:04:14.732Z,2026-04-09T18:04:14.732Z,billing_lambda,2026-04-10T21:03:44.988Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-09/evt_7024658bf3a44af9adde25d76268206c.json,2026-04-09,active


### Build Subscription State Table

In [0]:
silver_subscriptions_history_table_path = "s3://subscription-revenue-platform/silver/subscriptions_history/"
silver_subscriptions_history_checkpoint_path = "s3://subscription-revenue-platform/silver/subscriptions_history_checkpoints/"
silver_subscriptions_history_schema_path = f"{silver_subscriptions_history_checkpoint_path}/schema"

In [0]:
catalog = "subscription_platform"
schema = "silver"
silver_subscriptions_history_table = f"{catalog}.{schema}.subscriptions_history"

In [0]:
events_df = spark.read.table(silver_events_clean_table)

In [0]:
window_spec = Window.partitionBy("subscription_id").orderBy("event_time")

ordered_events = (
    events_df
    .withColumn("next_event_time", lead("event_time").over(window_spec))
)

In [0]:
history_df = (
    ordered_events
    .select(
        "subscription_id",
        "customer_id",
        "plan_id",
        "amount",
        "status",
        col("event_time").alias("start_date"),
        col("next_event_time").alias("end_date")
    )
)

In [0]:
history_df = history_df.withColumn(
    "is_current",
    col("end_date").isNull()
)

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "subscription_platform.silver.subscriptions_history"
)

(
    target.alias("target")
    .merge(
        history_df.alias("source"),
        """
        target.subscription_id = source.subscription_id
        AND target.start_date = source.start_date
        """
    )
    .whenMatchedUpdate(set={
        "end_date": "source.end_date",
        "is_current": "source.is_current",
        "plan_id": "source.plan_id",
        "amount": "source.amount",
        "status": "source.status"
    })
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql

SELECT *
FROM subscription_platform.silver.subscriptions_history
ORDER BY subscription_id, start_date;

subscription_id,customer_id,plan_id,amount,status,start_date,end_date,is_current
sub_1001,cust_501,basic,1000.0,cancelled,2026-04-06T18:04:20.449Z,2026-04-06T18:04:22.934Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-06T18:04:22.934Z,2026-04-07T18:04:17.657Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-07T18:04:17.657Z,2026-04-07T18:04:22.934Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-07T18:04:22.934Z,2026-04-09T18:03:46.374Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-09T18:03:46.374Z,2026-04-09T18:03:46.374Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-09T18:03:46.374Z,2026-04-09T18:04:14.732Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-09T18:04:14.732Z,2026-04-09T18:04:17.656Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-09T18:04:17.656Z,2026-04-09T18:04:17.657Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-09T18:04:17.657Z,2026-04-09T18:04:20.449Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-09T18:04:20.449Z,2026-04-09T18:04:24.986Z,false
